In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
HF_API_KEY = userdata.get('HUGGINGFACE_API_KEY')
os.environ["HUGGINGFACE_API_KEY"] = HF_API_KEY
serper_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serper_api_key
serp_api_key = userdata.get('SERPER_API_KEY')
os.environ["SERPER_API_KEY"] = serp_api_key

In [ ]:
# model reference
gptmodel_id = "gpt2"
mistralmodel_id = "mistralai/Ministral-3-3B-Instruct-2512"
llamamodel_id = "meta-llama/Llama-3.2-3b-Instruct"
llamachatmodel_id = "meta-llama/Llama-2-7b-chat-hf"

In [ ]:
# Install required packages
!pip install -q transformers datasets trl peft accelerate bitsandbytes

In [ ]:
# Import necessary libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model, PeftModel
import json

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cpu


In [ ]:
# Load Llama Model for fine-tuning
model_name = llamamodel_id

print(f"Loading base model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # Use half precision for memory efficiency
    device_map="auto"
)

# Explicitly disable gradient checkpointing on the model
model.gradient_checkpointing_disable()

print(f"\nModel loaded on device: {model.device}")
print(f"Model parameters: {model.num_parameters():,}")


Loading base model: meta-llama/Llama-3.2-3b-Instruct


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


Model loaded on device: cpu
Model parameters: 3,212,749,824


In [ ]:
# Load a tokenizer with built-in chat template support
tokenizer = AutoTokenizer.from_pretrained(llamamodel_id)

# Example conversation in standard format
messages = [
    {"role": "system", "content": "You are a helpful AI assistant specialized in medical question and answering."},
    {"role": "user", "content": "Hallmark of breast malignancy on mammography?"},
    {"role": "assistant", "content": "Clusters of microcalcification"},
    {"role": "user", "content": "Can you give me an example?"}
]

# Apply chat template to format the conversation
formatted_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,  # Return string, not token IDs
    add_generation_prompt=True  # Add prompt for assistant's next response
)

print("=" * 70)
print("FORMATTED CHAT WITH TEMPLATE:")
print("=" * 70)
print(formatted_chat)
print("=" * 70)

FORMATTED CHAT WITH TEMPLATE:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Feb 2026

You are a helpful AI assistant specialized in medical question and answering.<|eot_id|><|start_header_id|>user<|end_header_id|>

Hallmark of breast malignancy on mammography?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Clusters of microcalcification<|eot_id|><|start_header_id|>user<|end_header_id|>

Can you give me an example?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [ ]:
# Let's create a simple dataset with conversations
sample_conversations = [
    [
        {"role": "user", "content": "Enzyme used in PCR is ?"},
        {"role": "assistant", "content": "None"}
    ],
    [
        {"role": "user", "content": "Bruxism is -"},
        {"role": "assistant", "content": "Grinding of teeth during sleep"}
    ],
    [
        {"role": "user", "content": "Methysergide is banned as it causes?"},
        {"role": "assistant", "content": "Pulmonary fibrosis"}
    ]
]

# Create a dataset and apply chat template to each conversation
dataset = Dataset.from_dict({"messages": sample_conversations})

# Apply chat template formatting
def format_chat(example):
    return {
        "formatted_chat": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

formatted_dataset = dataset.map(format_chat)

# Display first example
print("Original conversation:")
print(json.dumps(sample_conversations[0], indent=2))
print("\n" + "=" * 70)
print("Formatted with chat template:")
print("=" * 70)
print(formatted_dataset[0]["formatted_chat"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Original conversation:
[
  {
    "role": "user",
    "content": "Enzyme used in PCR is ?"
  },
  {
    "role": "assistant",
    "content": "None"
  }
]

Formatted with chat template:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Enzyme used in PCR is ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

None<|eot_id|>


In [ ]:
# Configure SFT training
training_args = SFTConfig(
    output_dir="./sft_output",

    # Training hyperparameters
    num_train_epochs=1,  # Keep it short for demo
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8

    # Optimizer settings
    learning_rate=5e-5,
    warmup_steps=50,

    # Logging and evaluation
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=50,

    load_best_model_at_end = True, # REQUIRED for early stopping
    metric_for_best_model = "eval_loss", # Metric to monitor
    greater_is_better = False,

    # Memory optimization
    fp16=False,  # Use mixed precision training
    gradient_checkpointing=False, # Set to False to resolve ValueError
    use_cpu=True, # Explicitly tell SFTConfig to use CPU

    # SFT specific
    # max_seq_length=512, # This parameter is not accepted by SFTConfig in this version.
    packing=False  # Don't pack multiple examples together
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16 enabled: {training_args.fp16}")
print(f"  Gradient checkpointing: {training_args.gradient_checkpointing}")
print(f"  Use CPU: {training_args.use_cpu}")
# print(f"  Max sequence length: {training_args.max_seq_length}") # Uncommented print statement

Training configuration:
  Epochs: 1
  Batch size: 2
  Gradient accumulation: 4
  Effective batch size: 8
  Learning rate: 5e-05
  FP16 enabled: False
  Gradient checkpointing: False
  Use CPU: True


In [ ]:
# Load PubMedQA dataset
from datasets import load_dataset

#dataset = load_dataset("llamafactory/PubMedQA")
# or load the separate splits if the dataset has train/validation/test splits
train_dataset = load_dataset("llamafactory/PubMedQA", split="train")
test_dataset  = load_dataset("llamafactory/PubMedQA", split="test")

In [ ]:
import json

# Take a small subset for quick training (100 examples)
train_dataset = train_dataset.select(range(100))
eval_dataset = test_dataset.select(range(100, 125))  # 25 examples for validation

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")
print("\n" + "=" * 25)
print("Sample training example:")
print("=" * 25)
# Modified to print the entire first example to inspect its structure
print(json.dumps(train_dataset[0], indent=2))

Training examples: 100
Validation examples: 25

Sample training example:
{
  "instruction": "Answer the question based on the following context: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting. To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms. A retrospective cohort study, using abstracted data from medical charts. One natural medicine and six conventional medical clinics at Community Health Centers of King County, Washington, from November 1, 1996, through July 31, 1998. Women aged 40 years of age or more with a diagnosis of menopausal symptoms documented by a naturopathic or conventional physician. Improvement in selected menopausal symptoms. In univariate analyses, patients treated with naturopathy for menopausal sy

In [ ]:
# del model
torch.cuda.empty_cache()

In [ ]:
def format_pubmedqa_chat(example):
    user_content = f"{example['instruction']}\nQuestion: {example['input']}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example['output']}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

print("Function `format_pubmedqa_chat` defined.")

Function `format_pubmedqa_chat` defined.


In [ ]:
train_dataset = train_dataset.map(lambda x: {"text": format_pubmedqa_chat(x)}, batched=False) # Renamed to 'text'
eval_dataset = eval_dataset.map(lambda x: {"text": format_pubmedqa_chat(x)}, batched=False) # Renamed to 'text'

print("Formatted training example:")
print("=" * 70)
print(train_dataset[0]["text"]) # Changed to 'text'

Formatted training example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Answer the question based on the following context: Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting. To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms. A retrospective cohort study, using abstracted data from medical charts. One natural medicine and six conventional medical clinics at Community Health Centers of King County, Washington, from November 1, 1996, through July 31, 1998. Women aged 40 years of age or more with a diagnosis of menopausal symptoms documented by a naturopathic or conventional physician. Improveme

In [ ]:
# Initialize the trainer
from transformers import EarlyStoppingCallback
# Create SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)] # Stop after 3 checks
)

print("SFTTrainer created successfully!")
print(f"\nTotal training steps: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

SFTTrainer created successfully!

Total training steps: 12


In [ ]:
# Train the model (Llama-3.2-3b-Instruct) on PubMedQA dataset
print("Starting training...\n")
print("=" * 70)

trainer.train()

print("\n" + "=" * 70)
print("Training completed!")

# P.S. Error Message. Your session crashed after using all available RAM. Get more RAM

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Starting training...



In [ ]:
# Save the fine-tuned model
trainer.save_model("./sft_finetuned_model")
tokenizer.save_pretrained("./sft_finetuned_model")

print("Model saved to ./sft_finetuned_model")

In [ ]:
# PEFT with LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a fresh base model for LoRA training
print("Loading fresh base model for LoRA training...\n")

lora_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

lora_tokenizer = AutoTokenizer.from_pretrained(model_name)


# The function `clone_chat_template` is not defined in the notebook.
# If you intend to use a specific chat template function, please define it first.
# Removing this line to resolve the NameError.
# lora_model, lora_tokenizer, _ = clone_chat_template(lora_model, lora_tokenizer, model_name)


print(f"Base model loaded: {model_name}")
print(f"Total parameters: {lora_model.num_parameters():,}")
print(f"Chat template available: {lora_tokenizer.chat_template is not None}")

Loading fresh base model for LoRA training...



Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Base model loaded: meta-llama/Llama-3.2-3b-Instruct
Total parameters: 3,212,749,824
Chat template available: True


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=4,  # Rank - higher = more capacity but more parameters
    lora_alpha=8,  # Scaling factor (typically 2x rank)
    lora_dropout=0.1,  # Dropout for regularization
    bias="none",  # Don't train bias terms
    task_type="CAUSAL_LM",  # Causal language modeling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Apply to attention layers
)

# Apply LoRA to model
lora_model = get_peft_model(lora_model, lora_config)

# Print trainable parameters
lora_model.print_trainable_parameters()

print("\nLoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713

LoRA Configuration:
  Rank (r): 4
  Alpha: 8
  Target modules: {'v_proj', 'q_proj', 'k_proj', 'o_proj'}
  Dropout: 0.1


In [ ]:
# Configure training for LoRA
lora_training_args = SFTConfig(
    output_dir="./lora_output",

    # Training hyperparameters
    num_train_epochs=1,
    per_device_train_batch_size=2,  # Can use larger batch with LoRA!
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,

    # Optimizer settings
    learning_rate=3e-4,  # Can use higher LR with LoRA
    warmup_steps=50,

    # Logging and evaluation
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=50,

    load_best_model_at_end = True, # REQUIRED for early stopping
    metric_for_best_model = "eval_loss", # Metric to monitor
    greater_is_better = False,

    # Memory optimization
    # fp16=False,
    use_cpu=True,
    gradient_checkpointing=False,

    # SFT specific
    #max_seq_length=512,
    packing=False,
)

print("LoRA Training configuration:")
print(f"  Batch size: {lora_training_args.per_device_train_batch_size} (vs 2 for full FT)")
print(f"  Learning rate: {lora_training_args.learning_rate} (vs 5e-5 for full FT)")
print(f"  Memory: Lower due to frozen base model weights")

LoRA Training configuration:
  Batch size: 2 (vs 2 for full FT)
  Learning rate: 0.0003 (vs 5e-5 for full FT)
  Memory: Lower due to frozen base model weights


In [ ]:
# Create trainer for LoRA
from transformers import EarlyStoppingCallback # Added import statement
# Create SFTTrainer
lora_trainer = SFTTrainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=lora_tokenizer,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)] # Stop after 3 checks
)

print("LoRA SFTTrainer created successfully!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


LoRA SFTTrainer created successfully!


In [ ]:
# Train with LoRA
print("Starting LoRA training...\n")
print("=" * 25)

lora_trainer.train()

print("\n" + "=" * 25)
print("LoRA training completed!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Starting LoRA training...



In [ ]:
# Save LoRA adapters (much smaller than full model!)
lora_trainer.save_model("./lora_adapters")
lora_tokenizer.save_pretrained("./lora_adapters")

print("LoRA adapters saved to ./lora_adapters")
print("\nNote: LoRA adapters are typically only a few MBs,")
print("while full model checkpoints are hundreds of MBs!")

In [ ]:
# Test LoRA fine-tuned model
print("=" * 25)
print("LoRA FINE-TUNED MODEL:")
print("=" * 25)
print(f"Prompt: {test_prompt}\n")
print("Response:")
print(test_model(lora_model, lora_tokenizer, test_prompt, max_length=50))

print("\n" + "=" * 25)
print(f"Prompt: {test_prompt_2}\n")
print("Response:")
print(test_model(lora_model, lora_tokenizer, test_prompt_2, max_length=80))

In [ ]:
# Option 1: Merge adapters for deployment (no inference latency)
merged_model = model_with_adapters.merge_and_unload()

print("Adapters merged into base model!")
print("\nMerged model behaves exactly like the LoRA model,")
print("but without separate adapter layers (faster inference).")

# You can save this merged model
# merged_model.save_pretrained("./merged_model")

In [ ]:
# Test merged model
print("=" * 70)
print("MERGED MODEL (LoRA adapters merged):")
print("=" * 70)
print(f"Prompt: {test_prompt}\n")
print("Response:")
print(test_model(merged_model, lora_tokenizer, test_prompt, max_length=50))

In [ ]:
#

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/My Drive/

/content/drive/My Drive


In [ ]:
%cd /content/drive/My Drive/

/content/drive/My Drive


In [ ]:
import numpy as np
import pandas as pd


df_medreason = pd.read_json('medreason-instruction-dataset.json')
df_medreason.head(3)

,messages
0,"[{'role': 'user', 'content': 'Most common site..."
1,"[{'role': 'user', 'content': 'Most common caus..."
2,"[{'role': 'user', 'content': 'Most common orga..."
